# 03 — Eval

Phase 3 (Per-Field-Accuracy Baseline), Phase 4 (Iterations-Auswertung + Synthese-Tabelle), Phase 6 (Skalierung 7B + 3B-Halluzinations-Klassen).

Predictions kommen aus `02_extract.ipynb`. Gold ist eure `annotation/meine_gold.csv` aus Phase 2.

## Run-Header

| Feld | Wert |
|---|---|
| Datum | _YYYY-MM-DD_ |
| Gold-Datei | `annotation/meine_gold.csv` |
| Aktive Predictions-Datei(en) | _ |
| Match-Entscheidung `skills_top3` | _ (Set / geordnete Liste) |
| Match-Entscheidung `gehalt_min_eur` | _ (exakt / Toleranz X %) |
| JSON-Parse-Fails (Anzahl) | _ |

## Phase 3 — Baseline-Accuracy auf 12 Hand-Gold-Anzeigen

Hypothese-Cell *vor* der Eval: welches Feld haltet ihr für am stärksten / am schwächsten — und warum?

### Hypothese vor der Eval

Basierend auf dem κ-Befund aus Phase 2 und der Korpus-Inspektion:

- **Erwartet am stärksten:** `vertragsart` — die API liefert mit `stellenangebotsart` eine starke Vorinformation; selbst über den reinen Text sind Ausbildung / Festanstellung / Praktikum meist eindeutig. κ Mensch↔Mensch war hier 0.826.
- **Erwartet am schwächsten:** `homeoffice` — schon zwischen Menschen κ=0.122. Ambivalente Formulierungen wie *„Möglichkeit zum hybriden Arbeiten"*, *„Gleitzeit und Homeoffice"*, *„nach Absprache"* werden auch das Modell schwanken lassen, vor allem zwischen `ja` / `teilweise` / `nicht_genannt`.
- **Wildcard `gehalt_min_eur`:** in unserem Korpus haben ~80% gar keine €-Zahl im Text → viele Gold-`null`s → Feld wirkt künstlich „leicht", solange das Modell brav `null` schreibt. Echtes Signal kommt aus den 2–3 Anzeigen mit konkreter Zahl (z. B. Ausbildung mit „1.100 € monatlich").

**Match-Entscheidungen** (vorab dokumentieren, gehören in Stufe-3-Bewertung):
- `skills_top3`: **Set-Match** (case-insensitive). Reihenfolge ist subjektiv, ein Modell-Output `["SQL", "Python"]` und Gold `["Python", "SQL"]` zählen als Match.
- `gehalt_min_eur`: **Toleranz ±5%**. Eine Anzeige mit Range „50.000–60.000 €" kann je nach Lesart 50000 (Untergrenze) oder 55000 (Mitte) liefern; 5% deckt das ab, ohne grobe Halluzinationen zu kaschieren. `null` muss `null` matchen (kein Toleranz-Spielraum).
- `_parse_fail`: zählt für **alle 6 Felder** dieser Anzeige als Miss.

In [ ]:
import json
from pathlib import Path

import pandas as pd

PRED_PATH = Path("predictions.jsonl")              # aus 02_extract.ipynb (Phase 3)
GOLD_PATH = Path("../annotation/meine_gold.csv")

GEHALT_TOLERANCE = 0.05   # ±5 % bei gehalt_min_eur

# Predictions laden (eine Zeile pro Anzeige)
preds_by_id = {}
for line in PRED_PATH.open(encoding="utf-8"):
    obj = json.loads(line)
    preds_by_id[obj["refnr"]] = obj

# Gold laden
gold_df = pd.read_csv(GOLD_PATH)
gold_by_id = {str(row["id"]): dict(row) for _, row in gold_df.iterrows()}

common = sorted(set(preds_by_id) & set(gold_by_id))
parse_fails = sum(1 for r in common if preds_by_id[r].get("_parse_fail"))

print(f"Gemeinsame IDs : {len(common)}")
print(f"JSON-Parse-Fails: {parse_fails}/{len(common)}")

In [ ]:
# ── Normalizer + Match-Funktionen ─────────────────────────────────────────

def _norm_str(v):
    """Gold-Strings ('null', 'Jahr', leer) und Prediction-Strings auf gemeinsamen Form bringen."""
    if v is None:
        return None
    if isinstance(v, float) and pd.isna(v):
        return None
    s = str(v).strip().lower()
    return None if s in ("", "null", "none", "nan") else s


def _norm_int(v):
    """Gold-Zahlen ('null', '55000', NaN) und Prediction-Zahlen auf int|None bringen."""
    if v is None or (isinstance(v, float) and pd.isna(v)):
        return None
    if isinstance(v, bool):     # int + bool teilen sich Typen in Python — bool ausschließen
        return None
    if isinstance(v, int):
        return v
    s = str(v).strip()
    if s.lower() in ("", "null", "none", "nan"):
        return None
    try:
        return int(float(s))
    except (ValueError, TypeError):
        return None


def _norm_skill_set(v):
    """Pred-Liste oder Gold-Pipe-String → Set normalisierter Lower-Case-Strings."""
    if isinstance(v, list):
        return {str(x).strip().lower() for x in v if isinstance(x, str) and x.strip()}
    if isinstance(v, str) and v.strip():
        return {p.strip().lower() for p in v.split("|") if p.strip() and p.strip().lower() != "null"}
    return set()


def match_cat(p, g) -> bool:
    return _norm_str(p) == _norm_str(g)


def match_gehalt(p, g) -> bool:
    pi, gi = _norm_int(p), _norm_int(g)
    if pi is None and gi is None:
        return True
    if pi is None or gi is None:
        return False
    if gi == 0:
        return pi == 0
    return abs(pi - gi) / abs(gi) <= GEHALT_TOLERANCE


def match_skills(p, g) -> bool:
    return _norm_skill_set(p) == _norm_skill_set(g)


# ── Per-Field-Accuracy ────────────────────────────────────────────────────

FELDER = [
    ("homeoffice",      match_cat),
    ("vertragsart",     match_cat),
    ("erfahrungslevel", match_cat),
    ("gehalt_min_eur",  match_gehalt),
    ("gehalt_zeitraum", match_cat),
    ("skills_top3",     match_skills),
]

rows = []
detail = []   # für Fehler-Anschau später

for feld, matcher in FELDER:
    n_match = 0
    for rid in common:
        pred = preds_by_id[rid]
        gold = gold_by_id[rid]
        if pred.get("_parse_fail"):
            ok = False
        else:
            ok = matcher(pred.get(feld), gold.get(feld))
        if ok:
            n_match += 1
        else:
            detail.append({
                "refnr": rid, "feld": feld,
                "pred": pred.get(feld), "gold": gold.get(feld),
                "parse_fail": bool(pred.get("_parse_fail")),
            })
    rows.append({
        "feld": feld,
        "n_korrekt": n_match,
        "n_total": len(common),
        "accuracy": n_match / len(common),
    })

acc_df = pd.DataFrame(rows)
acc_df["accuracy %"] = (acc_df["accuracy"] * 100).round(0).astype(int).astype(str) + " %"

print("Per-Field-Accuracy (n=12, baseline 7B):\n")
print(acc_df[["feld", "n_korrekt", "n_total", "accuracy %"]].to_string(index=False))
print(f"\nJSON-Parse-Fails: {parse_fails}/{len(common)}  (zählen als Miss in allen Feldern)")

In [ ]:
# Fehler-Details: pro Feld alle Anzeigen, wo pred != gold — als Material für die Diagnose-Zelle unten
detail_df = pd.DataFrame(detail)
if not detail_df.empty:
    print("Fehler im Detail (pred ≠ gold):\n")
    for feld in [r["feld"] for r in rows]:
        sub = detail_df[detail_df["feld"] == feld]
        if sub.empty:
            continue
        print(f"── {feld}  ({len(sub)} Misses) ──")
        for _, r in sub.iterrows():
            tag = " [PARSE_FAIL]" if r["parse_fail"] else ""
            print(f"  {r['refnr']}{tag}: pred={r['pred']!r:30s}  gold={r['gold']!r}")
        print()
else:
    print("Keine Misses (perfekte Pipeline 🎯 — unwahrscheinlich, prüfe Match-Logik).")

### Schwächste Felder + Hypothese (Schema / Modell / Pipeline)

Aus der Tabelle und der Fehler-Liste oben die **zwei schwächsten Felder** identifizieren und pro Feld eine Fehler-Klasse zuordnen.

| Feld | Accuracy (n_korrekt/12) | Top-Fehler-Beispiele (refnr) | Diagnose | Begründung am Text |
|---|---|---|---|---|
| _Feld 1_ | _X/12_ | _refnr 1, refnr 2_ | _Schema_ / _Modell_ / _Pipeline_ | _z. B. „Text sagt klar 'remote', Modell schreibt 'nicht_genannt' → Modell-Problem"_ |
| _Feld 2_ | _X/12_ | _ | _ | _ |

**Fehler-Klassen-Definition (aus Phase 3 Aufgabenblatt):**
- **Schema-Problem** — Wert im Text klar, aber Schema fängt ihn nicht sauber ab (z. B. „remote nach Absprache" passt nicht eindeutig in `{ja, teilweise, nein, nicht_genannt}`)
- **Modell-Problem** — Wert im Text klar + schemakonform extrahierbar, aber Modell wählt falsch (z. B. ignoriert „ab 50.000 €")
- **Pipeline-Problem** — Wert im Text, aber außerhalb des truncierten Kontexts (z. B. Gehalt erst nach 2500 Zeichen)

Diese zwei Diagnosen sind die **Hypothese für Phase 4 Iteration A + B**. Pro Iteration einen Hebel ziehen, der genau diese Fehler-Klasse adressiert (Prompt-Klarstellung, Few-Shot, Truncation, Modellgröße).

## Phase 4 — Iteration A Auswertung

Hypothese-Cell *vor* der Iteration: welches Feld, welche Δ-Größe, warum?

## Phase 4 — Iteration B Auswertung

Hypothese-Cell *vor* der Iteration.

## Phase 4 — Synthese

Iterations-Tabelle (Baseline / A / B mit Hypothese, Aktion, Δ Gesamt, Δ schwächstes Feld, Diagnose) + Synthese-Antworten zu den drei Fragen aus dem Aufgabenblatt.

## Phase 6 — Vollständiger 7B-Run + 3B-Halluzinations-Klassen

Per-Field-Accuracy auf den 12 Hand-Gold-Anzeigen + Schema-Konformitäts-Check auf den restlichen Anzeigen ohne Gold. 3B-vs-7B-vs-Gold per `refnr` joinen, drei eigenständige Halluzinations-Klassen mit konkreten Beispielen identifizieren.